# 🔌CONEXÃO
Estabelecer conexão com os bancos

In [1]:
import psycopg2
from psycopg2.extras import DictCursor
import fdb
from dotenv import load_dotenv
import os

load_dotenv()

# Conexões
cnx_dest = fdb.connect(
    user=os.getenv("FDB_USER"),
    password=os.getenv("FDB_PASS"),
    host=os.getenv("FDB_HOST"),
    port=int(os.getenv("FDB_PORT")),
    database=os.getenv("FDB_PATH"),
    charset="WIN1252"
)
cur_dest = cnx_dest.cursor()

cnx_orig = psycopg2.connect(
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASS"),
    host=os.getenv("PG_HOST"),
    database=os.getenv("PG_DB_PATRI")
    #options="-c search_path={}".format(os.getenv("PG_SCHEMA"))
)
cnx_orig.autocommit = True
cur_orig = cnx_orig.cursor(cursor_factory=DictCursor)

# cnx_arq = fdb.connect(
#     user=os.getenv("FDB_USER"),
#     password=os.getenv("FDB_PASS"),
#     host=os.getenv("FDB_HOST"),
#     port=int(os.getenv("FDB_PORT")),
#     database=os.getenv("FDB_PATH_ARQ"),
#     charset="WIN1252"
# )
# cur_arq = cnx_arq.cursor()

def commit():
    cnx_dest.commit()

base_dir = os.getcwd()  # diretório de trabalho atual
base_dir = os.path.dirname(base_dir)
pasta_arquivos = os.path.join(base_dir, "arquivos")

# 🛠️ FERRAMENTAS
Funções, variáveis cache, hashmaps

In [2]:
global cadest, empresa, exercicio
cadest = {}
empresa = cur_dest.execute("SELECT empresa FROM cadcli").fetchone()[0]
exercicio = cur_dest.execute("SELECT mexer FROM cadcli").fetchone()[0]

def limpa_tabela(tabelas):
    for tabela in tabelas:
        cur_dest.execute(f"DELETE FROM {tabela}")
    commit()

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

def to_cp1252_safe(value, replace_with=''):
    """
    Converte uma string para cp1252 de forma segura.
    Remove ou substitui caracteres inválidos.

    :param value: valor a converter
    :param replace_with: string usada no lugar de caracteres inválidos (default = '')
    :return: valor limpo
    """
    if isinstance(value, str):
        try:
            # Tenta converter diretamente
            value.encode('cp1252')
            return value
        except UnicodeEncodeError:
            # Remove/substitui caracteres inválidos
            return value.encode('cp1252', errors='replace').decode('cp1252').replace('?', replace_with)
    return value

# 🏛️ PATRIMÔNIO
Extração, tratamento e carregamento dos dados referentes ao módulo patrimônio

## CADASTROS BASE

### TIPO DE MOVIMENTAÇÕES

In [3]:
limpa_tabela(("pt_tipomov",))

valores = {
    "A": "Aquisição",
    "B": "Baixa",
    "T": "Transferência",
    "R": "Procedimento Contábil",
    "P": "Transferência de Plano Contábil",
}

cur_dest.executemany("INSERT INTO PT_TIPOMOV (codigo_tmv, descricao_tmv) VALUES (?, ?)", valores.items())

commit()

### TIPOS DE BAIXA

In [4]:
limpa_tabela(("pt_cadbai",))

insert = cur_dest.prep("INSERT INTO PT_CADBAI (CODIGO_BAI, EMPRESA_BAI, DESCRICAO_BAI) VALUES (?, ?, ?)")

cur_orig.execute("""
select
	identificador_de_registro::int codigo_bai,
	cdtdb.codigo_do_entidade::int empresa_bai,
	cdtdb.descricao_do_tipo_de_baixa descricao_bai
from
	cadastro_dos_tipos_de_baixas cdtdb
""")

for row in cur_orig:
    cur_dest.execute(insert, (row['codigo_bai'], row['empresa_bai'], row['descricao_bai']))
commit()

### CONSERVAÇÃO

In [5]:
limpa_tabela(("pt_cadsit",))

insert = cur_dest.prep("INSERT INTO PT_CADSIT (CODIGO_SIT, EMPRESA_SIT, DESCRICAO_SIT) VALUES (?, ?, ?)")

cur_orig.execute("select ('141'||identificador_de_registro)::int identificador_de_registro, descricao_do_tipo_de_obra from cadastro_dos_estados_de_conservacao cdedc")

for row in cur_orig:
    cur_dest.execute(insert, (row['identificador_de_registro'], empresa, row['descricao_do_tipo_de_obra']))
commit()

cur_dest.execute("""
INSERT INTO PT_CADSIT (CODIGO_SIT, EMPRESA_SIT, DESCRICAO_SIT)
SELECT t.EMPRESA||substring(CODIGO_SIT FROM char_length(empresa) for 20) codigo_sit, EMPRESA,DESCRICAO_SIT FROM pt_cadsit a, tabempresa t 
WHERE NOT EXISTS (SELECT 1 FROM pt_cadsit x WHERE x.CODIGO_SIT = a.CODIGO_SIT AND t.EMPRESA = a.EMPRESA_SIT)
""")
commit()

### TIPOS DE BEM

In [6]:
limpa_tabela(("pt_cadtip",))

insert = cur_dest.prep("INSERT INTO PT_CADTIP (CODIGO_TIP, EMPRESA_TIP, DESCRICAO_TIP) VALUES (?, ?, ?)")

cur_orig.execute("""
select
	identificador::int codigo_tip,
	entidade::int empresa_tip,
	descricao_do_metodo_de_depreciacao descricao_tip
from
	cadastro_dos_grupos_de_bens a
""")

for row in cur_orig:
    cur_dest.execute(insert, (row['codigo_tip'], row['empresa_tip'], row['descricao_tip'][:60]))
commit()

### UNIDADE E SUBUNIDADE

In [7]:
limpa_tabela(("pt_cadpats", "pt_cadpatd"))

insert = cur_dest.prep("INSERT INTO PT_CADPATD (EMPRESA_DES, CODIGO_DES, NAUNI_DES, OCULTAR_DES) values (?, ?, ?, 'N')")
insert_cadpats = cur_dest.prep(f"INSERT INTO PT_CADPATS (codigo_set, empresa_set, codigo_des_set, noset_set, ocultar_set) VALUES (?, ?, ?, ?, 'N')")

cur_orig.execute('''
select
	cdlfdb.entidade::int empresa,
	identificador::int setor,
	identificador_da_localizacao_fisica_pai::int pai,
	substr(cdlfdb.descricao_da_localizacao_fisica, 1, 60) nome,
	localizacao_fisica_ativa_ou_inativa,
	nivel_da_localizacao_fisica
from
	cadastro_de_localizacoes_fisicas_dos_bens cdlfdb
order by 1,2
''')

# with unidade as (
# select
# 	dense_rank() over (order by numero_que_representa_o_organograma) codigo_des,
# 	oc.numero_que_representa_o_organograma codant,
# 	max(oc.descricao) descr,
# 	max(codigo_dos_organogramas_contratos) id
# from
# 	organogramas_contratos oc
# where nivel <= 2
# group by
# 	2),
# subunidade as (select
# 	oc.numero_que_representa_o_organograma codant,
# 	max(oc.descricao) descr,
# 	max(codigo_dos_organogramas_contratos) id
# from
# 	organogramas_contratos oc
# where nivel = 3
# group by
# 	1)
# select codigo_des setor, null pai, descr, codant from unidade 
# union all
# select row_number() over (order by s.codant) setor, u.codigo_des pai, s.descr nome, s.codant from subunidade s
# join unidade u on substr(s.codant,1,5) = substr(u.codant,1,5)
    
for row in cur_orig:
    if row['pai'] is None:
        cur_dest.execute(insert, (row['empresa'], row['setor'], row['nome']))
        cur_dest.execute(insert_cadpats, (row['setor'], row['empresa'], row['setor'], row['nome']))
        commit()
    else:
        cur_dest.execute(insert_cadpats, (row['setor'], row['empresa'], row['pai'], row['nome']))
commit()

cur_dest.execute("""
INSERT INTO pt_cadpatd (EMPRESA_DES, CODIGO_DES, NAUNI_DES, OCULTAR_DES) SELECT empresa, empresa||0, 'NÃO INFORMADO','N' FROM tabempresa t
WHERE NOT EXISTS (SELECT 1 FROM pt_cadpatd b WHERE codigo_des = t.empresa||0)
""")
commit()

cur_dest.execute("""
INSERT INTO pt_cadpats (codigo_set, empresa_set, codigo_des_set, noset_set, ocultar_set) SELECT empresa||0, empresa, empresa||0, 'NÃO INFORMADO','N' FROM tabempresa t
WHERE NOT EXISTS (SELECT 1 FROM pt_cadpats b WHERE codigo_set = t.empresa||0)
""")
commit()

### GRUPO

In [8]:
limpa_tabela(("pt_cadpatg",))
cria_coluna("pt_cadpatg", "codant")

cur_orig.execute("""
select
    dense_rank() over (order by identificador) codigo_gru,
    cdtdb.entidade,
    cdtdb.descricao_do_tipo_do_bem nogru_gru,
    identificador codant
from
    cadastro_dos_tipos_de_bem cdtdb
""")

for row in cur_orig:
    try:
        cur_dest.execute(f"INSERT INTO PT_CADPATG (CODIGO_GRU, EMPRESA_GRU, NOGRU_GRU, CODANT) VALUES ({row['codigo_gru']}, {row['entidade']}, '{row['nogru_gru']}', '{row['codant']}')")
    except:
        print(f"Erro ao inserir grupo {row['codigo_gru']} - {row['entidade']} - {row['nogru_gru']}")
commit()

### RESPONSAVEIS

In [9]:
limpa_tabela(("pt_cadresponsavel",))
cria_coluna("pt_cadresponsavel", "codant")

cur_orig.execute("""select distinct
	dense_rank() over (order by identificador_de_registro) codigo_resp,
	p.identificador_de_registro,
	p.nome,
	p.cpf_cnpj cpf
from
	pessoas p
join bens b on b.codigo_do_responsavel = p.identificador_de_registro""")

for row in cur_orig:
    cur_dest.execute(f"INSERT INTO PT_CADRESPONSAVEL (CODIGO_RESP, NOME_RESP, CPF_RESP, CODANT) VALUES ({row['codigo_resp']}, '{to_cp1252_safe(row['nome'])}', '{row['cpf']}', '{row['identificador_de_registro']}')")
commit()

## BENS

In [10]:
limpa_tabela(("pt_movbem","pt_cadpat"))

insert = cur_dest.prep(f"""
insert into pt_cadpat (
    codigo_pat,
    empresa_pat,
    codigo_gru_pat,
    chapa_pat,
    codigo_cpl_pat,
    codigo_set_pat,
    codigo_set_atu_pat,
    orig_pat,
    codigo_tip_pat,
    codigo_sit_pat,
    discr_pat,
    datae_pat,
    dtlan_pat,
    valaqu_pat,
    valatu_pat,
    codigo_for_pat,
    percenqtd_pat,
    dae_pat,
    valres_pat,
    percentemp_pat,
    nempg_pat,
    anoemp_pat,
    dtpag_pat,
    hash_sinc,
    codigo_bai_pat,
    chapa_pat_alt,
    serie_pat,
    nota_pat,
    processo_pat,
    codigo_resp_pat,
    obs_pat)
VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute("""
with locais as (
select
	codigo_do_bem,
	t.codigo_da_localizacao_fisica dest,
	tbo.codigo_da_localizacao_fisica_do_bem orig,
	row_number() over (partition by codigo_do_bem order by tb.codigo_da_transferencia asc) seq
from
	transferencia_bens tb
join tranferencias t on tb.codigo_da_transferencia = t.codigo_do_inventario
join transferencia_bens_orig tbo on tb.codigo_do_bem_da_transferencia = tbo.codigo_do_bem_da_transferencia
where codigo_da_localizacao_fisica_do_bem is not null),
grupos as (select
    dense_rank() over (order by identificador) codigo_gru,
    cdtdb.entidade,
    cdtdb.descricao_do_tipo_do_bem nogru_gru,
    identificador codant
from
    cadastro_dos_tipos_de_bem cdtdb),
baixas as (select
	bb.codigo_do_bem,
	b.data_e_hora_da_baixa::date dtpag_pat,
	b.codigo_do_tipo_de_baixa codigo_bai_pat,
	row_number() over (partition by bb.codigo_do_bem order by bb.identificador_de_registro desc) rn
from
	baixas_bens bb
join cadastro_de_baixas b on bb.codigo_da_baixa = b.identificador_de_registro),
responsaveis as (
                 select distinct
	dense_rank() over (order by identificador_de_registro) codigo_resp,
	p.identificador_de_registro,
	p.nome,
	p.cpf_cnpj cpf
from
	pessoas p
join bens b on b.codigo_do_responsavel = p.identificador_de_registro
)
select
	codigo_do_encerramento_do_periodo::int codigo_pat,
	b.codigo_da_entidade::int empresa_pat,
	g.codigo_gru::int,
	to_char(row_number() over (order by codigo_do_encerramento_do_periodo), 'fm000000') chapa_pat,
	null codigo_cpl_pat,
	coalesce(orig, concat(b.codigo_da_entidade,0)::numeric)::int codigo_set_pat, --coalesce(orig, coalesce(b.codigo_da_localizacao_fisica, concat(b.codigo_da_entidade,0)::numeric)::int)::int codigo_set_pat,
	coalesce(b.codigo_da_localizacao_fisica, concat(b.codigo_da_entidade,0)::numeric)::int codigo_set_atu_pat,
	case when classificacao_do_tipo_de_aquisicao = 'M' then 'O' when classificacao_do_tipo_de_aquisicao = 'O' then 'I' else classificacao_do_tipo_de_aquisicao end orig_pat,
	b.codigo_do_grupo_do_bem::int codigo_tip_pat,
	case when b.codigo_da_entidade <> 538 then concat(b.codigo_da_entidade, codigo_da_conservacao)::int else codigo_da_conservacao::int end codigo_sit_pat,
	b.descricao_do_bem discr_pat,
	b.data_de_aquisicao_do_bem::date datae_pat,
	coalesce(data_de_inicio_da_depreciacao::date, b.data_de_aquisicao_do_bem::date) dtlan_pat,
	v.valor_convertido_da_aquisicao valaqu_pat,
	v.valor_liquido_contabil valatu_pat,
	b.codigo_do_fornecedor::int codigo_for_pat,
	(v.quantidade_em_anos_de_vida_util*12)::int percenqtd_pat,
	case when quantidade_em_anos_de_vida_util is not null then 'V' else 'N' end dae_pat,
	v.valor_residual valres_pat,
	case when quantidade_em_anos_de_vida_util is not null then 'M' else null end percentemp_pat,
    nullif(split_part(numero_e_ano_do_empenho, '/', 1),'')::int AS nempg_pat,
    substr(case when nullif(split_part(numero_e_ano_do_empenho, '/', 2),'') not like '20%' then to_char(nullif(split_part(numero_e_ano_do_empenho, '/', 2),'')::int, 'fm2000') else  nullif(split_part(numero_e_ano_do_empenho, '/', 2),'') end,1,4)::int AS anoemp_pat,
	b.numero_de_registro_do_bem::bigint codigo_ant_pat,
	numero_da_placa chapa_pat_alt,
	bx.dtpag_pat,
	bx.codigo_bai_pat::int,
    b.numero_do_comprovante nota_pat,
    numero_e_ano_do_processo_administrativo processo_pat,
    r.codigo_resp,
    b.observacao obs_pat
from
	bens b
join valores_dos_bens v on v.codigo_do_bem = b.codigo_do_encerramento_do_periodo
left join locais l on l.codigo_do_bem = codigo_do_encerramento_do_periodo and l.seq = 1
left join cadastro_dos_tipos_de_aquisicao cdt on b.codigo_do_tipo_de_aquisicao = cdt.identificador_de_registro
left join grupos g on g.codant = codigo_do_tipo_do_bem
left join baixas bx on bx.codigo_do_bem = b.codigo_do_encerramento_do_periodo and bx.rn = 1
left join responsaveis r on r.identificador_de_registro = b.codigo_do_responsavel
--where b.codigo_da_entidade = 538
order by 1
""")

for row in cur_orig:
    try:
        cur_dest.execute(insert, (
            row['codigo_pat'],
            row['empresa_pat'],
            row['codigo_gru'],
            row['chapa_pat'],
            row['codigo_cpl_pat'],
            row['codigo_set_pat'],
            row['codigo_set_atu_pat'],
            row['orig_pat'],
            row['codigo_tip_pat'],
            row['codigo_sit_pat'],
            row['discr_pat'][:255],
            row['datae_pat'],
            row['dtlan_pat'],
            row['valaqu_pat'],
            row['valatu_pat'],
            row['codigo_for_pat'],
            row['percenqtd_pat'],
            row['dae_pat'],
            row['valres_pat'],
            row['percentemp_pat'],
            row['nempg_pat'],
            row['anoemp_pat'],
            row['dtpag_pat'],
            row['codigo_pat'],
            row['codigo_bai_pat'],
            row['chapa_pat_alt'],
            row['codigo_ant_pat'],
            row['nota_pat'],
            row['processo_pat'],
            row['codigo_resp'],
            row['obs_pat']
        ))
    except Exception as e:
        print(f"Erro ao inserir patrimônio {row['codigo_pat']}: {e}")
commit()

# cur_dest.execute("update pt_cadpat set chapa_pat = lpad(chapa_pat_alt, 6, 0) where chapa_pat_alt is not null and CHAR_LENGTH (chapa_pat_alt) <= 6")
# commit()

cur_dest.execute("""
MERGE INTO pt_cadpat a USING (SELECT codif, codif_ant FROM desfor) b
ON a.CODIGO_FOR_PAT = b.codif_ant
WHEN MATCHED THEN UPDATE SET a.CODIGO_FOR_PAT = b.codif
""")
commit()

cur_dest.execute("""
UPDATE pt_cadpat SET chapa_pat =  lpad(replace(replace(replace(trim(chapa_pat_alt),'.',''), 'b',''),'/',''), 6, '0')
""")
commit()

### MOVIMENTAÇÃO

In [11]:
limpa_tabela(("pt_movbem",))
cria_coluna("pt_movbem", "codigo_set_ant_mov")

insert = cur_dest.prep("""
INSERT
	INTO
		pt_movbem (codigo_mov,
		empresa_mov,
		codigo_pat_mov,
		data_mov,
		tipo_mov,
		codigo_set_mov,
		historico_mov,
		codigo_cpl_mov,
		codigo_bai_mov,
		valor_mov,
		depreciacao_mov,
		codigo_set_ant_mov,
		dt_contabil,
		lote_mov)
	VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute("""
select 
		row_number() over () codigo_mov,
		empresa_mov::int,
		codigo_pat_mov::int,
		data_mov,
		tipo_mov,
		codigo_set_mov::int,
		historico_mov,
		codigo_cpl_mov,
		codigo_bai_mov::int,
		valor_mov,
		depreciacao_mov,
		codigo_set_ant_mov,
		data_mov dt_contabil,
		lote_mov::int			 
from (
with CADPAT as (
	with locais as (
	select
		codigo_do_bem,
		t.codigo_da_localizacao_fisica dest,
		tbo.codigo_da_localizacao_fisica_do_bem orig,
		row_number() over (partition by codigo_do_bem order by tb.codigo_da_transferencia asc) seq
	from
		transferencia_bens tb
	join tranferencias t on tb.codigo_da_transferencia = t.codigo_do_inventario
	join transferencia_bens_orig tbo on tb.codigo_do_bem_da_transferencia = tbo.codigo_do_bem_da_transferencia
	where codigo_da_localizacao_fisica_do_bem is not null),
	grupos as (select
	    dense_rank() over (order by identificador) codigo_gru,
	    cdtdb.entidade,
	    cdtdb.descricao_do_tipo_do_bem nogru_gru,
	    identificador codant
	from
	    cadastro_dos_tipos_de_bem cdtdb),
	baixas as (select
		bb.codigo_do_bem,
		b.data_e_hora_da_baixa::date dtpag_pat,
		b.codigo_do_tipo_de_baixa codigo_bai_pat
	from
		baixas_bens bb
	join cadastro_de_baixas b on bb.codigo_da_baixa = b.identificador_de_registro ),
	responsaveis as (
	                 select distinct
		dense_rank() over (order by identificador_de_registro) codigo_resp,
		p.identificador_de_registro,
		p.nome,
		p.cpf_cnpj cpf
	from
		pessoas p
	join bens b on b.codigo_do_responsavel = p.identificador_de_registro
	)
	select
		codigo_do_encerramento_do_periodo::int codigo_pat,
		b.codigo_da_entidade::int empresa_pat,
		g.codigo_gru::int,
		to_char(row_number() over (order by codigo_do_encerramento_do_periodo), 'fm000000') chapa_pat,
		null codigo_cpl_pat,
		coalesce(orig, concat(b.codigo_da_entidade,0)::numeric)::int codigo_set_pat,
		coalesce(b.codigo_da_localizacao_fisica, concat(b.codigo_da_entidade,0)::numeric)::int codigo_set_atu_pat,
		case when classificacao_do_tipo_de_aquisicao = 'M' then 'O' when classificacao_do_tipo_de_aquisicao = 'O' then 'I' else classificacao_do_tipo_de_aquisicao end orig_pat,
		b.codigo_do_grupo_do_bem::int codigo_tip_pat,
		codigo_da_conservacao::int codigo_sit_pat,
		b.descricao_do_bem discr_pat,
		b.data_de_aquisicao_do_bem::date datae_pat,
		coalesce(data_de_inicio_da_depreciacao::date, b.data_de_aquisicao_do_bem::date) dtlan_pat,
		v.valor_convertido_da_aquisicao valaqu_pat,
		v.valor_liquido_contabil valatu_pat,
		b.codigo_do_fornecedor::int codigo_for_pat,
		(v.quantidade_em_anos_de_vida_util*12)::int percenqtd_pat,
		case when quantidade_em_anos_de_vida_util is not null then 'V' else 'N' end dae_pat,
		v.valor_residual valres_pat,
		case when quantidade_em_anos_de_vida_util is not null then 'M' else null end percentemp_pat,
	    nullif(split_part(numero_e_ano_do_empenho, '/', 1),'')::int AS nempg_pat,
	    substr(case when nullif(split_part(numero_e_ano_do_empenho, '/', 2),'') not like '20%' then to_char(nullif(split_part(numero_e_ano_do_empenho, '/', 2),'')::int, 'fm2000') else  nullif(split_part(numero_e_ano_do_empenho, '/', 2),'') end,1,4)::int AS anoemp_pat,
		b.numero_de_registro_do_bem::bigint codigo_ant_pat,
		numero_da_placa chapa_pat_alt,
		bx.dtpag_pat,
		bx.codigo_bai_pat::int,
	    b.numero_do_comprovante nota_pat,
	    numero_e_ano_do_processo_administrativo processo_pat,
	    r.codigo_resp,
	    b.observacao obs_pat
	from
		bens b
	join valores_dos_bens v on v.codigo_do_bem = b.codigo_do_encerramento_do_periodo
	left join locais l on l.codigo_do_bem = codigo_do_encerramento_do_periodo and l.seq = 1
	left join cadastro_dos_tipos_de_aquisicao cdt on b.codigo_do_tipo_de_aquisicao = cdt.identificador_de_registro
	left join grupos g on g.codant = codigo_do_tipo_do_bem
	left join baixas bx on bx.codigo_do_bem = b.codigo_do_encerramento_do_periodo
	left join responsaveis r on r.identificador_de_registro = b.codigo_do_responsavel),
ORIGEM as (
	select
		codigo_do_bem_da_transferencia,
		codigo_da_localizacao_fisica_do_bem
	from
		transferencia_bens_orig tbo)
select
	codigo_do_movimento_do_bem codigo_mov,
	b.empresa_pat::int empresa_mov,
	codigo_do_bem codigo_pat_mov,
	bm.data_hora_da_reavaliacao::date data_mov,
	'A' tipo_mov,
	codigo_set_pat codigo_set_mov,
	'AQUISICAO' historico_mov,
	null codigo_cpl_mov,
	null codigo_bai_mov,
	valor_do_movimento valor_mov,
	'N' depreciacao_mov,
	null codigo_set_ant_mov,
	null lote_mov
from
	bens_movimentos bm
join CADPAT b on bm.codigo_do_bem = b.codigo_pat
where bm.origem_do_movimento  = 1
union all
select
	tb.codigo_da_transferencia codigo_mov,
	t.codigo_da_entidade::int entidade,
	tb.codigo_do_bem codigo_pat_mov,
	t.data_e_hora_da_transferencia::date data_mov,
	'T' tipo_mov,
	t.codigo_da_localizacao_fisica codigo_set_mov,
	'TRANSFERENCIA' historico_mov,
	null codigo_cpl_mov,
	null codigo_bai_mov,
	0 valor_mov,
	'N' depreciacao_mov,
	coalesce(o.codigo_da_localizacao_fisica_do_bem, (t.codigo_da_entidade::varchar||0)::int)::int codigo_set_ant_mov,
	t.numero_da_transferencia lote_mov
from
	transferencia_bens tb
join tranferencias t on
	t.codigo_do_inventario = tb.codigo_da_transferencia and t.codigo_do_tipo_de_transferencia in (4)
left join origem o on
	o.codigo_do_bem_da_transferencia = tb.codigo_do_bem_da_transferencia
union all
select
	bdd.codigo_da_depreciacao_1 codigo_mov,
	d.codigo_da_entidade::int empresa_mov,
	codigo_do_bem codigo_pat_mov,
	d.data_e_hora_da_depreciacao::date data_mov,
	'R' tipo_mov,
	null codigo_set_mov,
	'DEPRECIACAO' historico_mov,
	null codigo_cpl_mov,
	null codigo_bai_mov,
	-bdd.valor_depreciado,
	'S' depreciacao_mov,
	null codigo_set_ant_mov,
	d.numero_do_lote_de_depreciacao lote_mov
from
	bens_da_depreciacao bdd
join depreciacoes d on bdd.codigo_da_depreciacao_1 = d.codigo_da_depreciacao
union all
select
	codigo_do_relacionamento codigo_mov,
	r.codigo_da_entidade::int empresa_mov,
	rdb.codigo_do_bem codigo_pat_mov,
	r.data_e_hora_da_reavaliacao::date data_mov,
	'R' tipo_mov,
	null codigo_set_mov,
	'REAVALIAÇÃO' historico_mov,
	null codigo_cpl_mov,
	null codigo_bai_mov,
	rdb.diferencia_do_valor_atual_com_o_anterior valor_mov,
	'N' depreciacao_mov,
	null,
	r.numero_do_lote lote_mov
from
	reavaliacao_dos_bens rdb
join reavaliacoes r on
	r.codigo_do_relacionamento = rdb.codigo_da_reavaliacao) movimento
--where empresa_mov = 538
order by codigo_pat_mov, data_mov, case when tipo_mov = 'A' then 1 when tipo_mov = 'T' then 2 when tipo_mov = 'R' and depreciacao_mov = 'N' then 3 when tipo_mov = 'R' and depreciacao_mov = 'S' then 4 else 5 end
""")

batch = []
batch_size = 10000
for row in cur_orig:
	batch.append((
		row['codigo_mov'],
		row['empresa_mov'],
		row['codigo_pat_mov'],
		row['data_mov'],
		row['tipo_mov'],
		row['codigo_set_mov'],
		to_cp1252_safe(row['historico_mov']),
		row['codigo_cpl_mov'],
		row['codigo_bai_mov'],
		row['valor_mov'],
		row['depreciacao_mov'],
		row['codigo_set_ant_mov'],
		row['data_mov'],
		row['lote_mov']
	))
	if len(batch) >= batch_size:
		cur_dest.executemany(insert, batch)
		batch = []
		commit()
if batch:
	cur_dest.executemany(insert, batch)
commit()

cur_dest.execute("""MERGE INTO pt_cadpat a USING (select codigo_pat_mov, sum(valor_mov) valor from pt_movbem group by 1) b
ON a.codigo_pat = b.codigo_pat_mov 
WHEN MATCHED THEN UPDATE SET a.valatu_pat = b.valor""")
commit()

codigo_mov = cur_dest.execute("SELECT COALESCE(MAX(codigo_mov),0) FROM pt_movbem").fetchone()[0]

rows = cur_dest.execute("""
SELECT
	codigo_pat_mov,
	-sum(valor_mov) valor_mov,
	dtpag_pat data_mov,
	codigo_bai_pat codigo_bai_mov,
	codigo_set_atu_pat codigo_set_mov,
	empresa_pat empresa_mov
FROM
	pt_cadpat a
JOIN pt_movbem b ON a.codigo_pat = b.codigo_pat_mov AND a.DTPAG_PAT IS NOT null
GROUP BY 1,3,4,5,6
""").fetchall()

for row in rows:
	codigo_mov += 1
	cur_dest.execute(insert, (
		codigo_mov,
		row[5],
		row[0],
		row[2],
		'B',
		row[4],
		'BAIXA',
		None,
		row[4],
		row[1],
		'N',
		None,
		row[2],
		None
	))
commit()

# cur_dest.execute("""
# INSERT
# 	INTO
# 		pt_movbem (codigo_mov,
# 		empresa_mov,
# 		codigo_pat_mov,
# 		data_mov,
# 		tipo_mov,
# 		codigo_set_mov,
# 		historico_mov,
# 		codigo_cpl_mov,
# 		codigo_bai_mov,
# 		valor_mov,
# 		depreciacao_mov,
# 		codigo_set_ant_mov,
# 		dt_contabil,
# 		lote_mov)
# SELECT gen_id(gen_pt_movbem_id, 1),
# 	empresa_pat,
# 	codigo_pat,
# 	dtpag_pat,
# 	'B',
# 	codigo_set_atu_pat,
# 	'BAIXA',
# 	NULL,
# 	codigo_bai_pat,
# 	-valatu_pat,
# 	'N',
# 	NULL,
# 	dtpag_pat,
# 	NULL
# FROM pt_cadpat
# WHERE DTPAG_PAT IS NOT NULL
# """)

cur_dest.execute("""
EXECUTE BLOCK AS
BEGIN
	update pt_cadpat a set codigo_set_pat = codigo_set_atu_pat WHERE CODIGO_SET_PAT = empresa_pat||'0' AND CODIGO_SET_ATU_PAT <> empresa_pat||'0' AND 
	NOT EXISTS (SELECT 1 FROM pt_movbem b WHERE a.codigo_pat = b.codigo_pat_mov AND tipo_mov = 'T' AND a.CODIGO_SET_ATU_PAT = b.CODIGO_SET_MOV);
	update pt_movbem set codigo_set_mov = (select codigo_set_pat from pt_cadpat where codigo_pat = codigo_pat_mov and codigo_set_pat <> codigo_set_mov) where tipo_mov = 'A';
END
""")

### ANEXOS

In [12]:
cur_arq.execute("delete from PATR_ARQUIVOS")
cur_arq.execute("delete from PATR_FOTOS")
cnx_arq.commit()

cur_orig.execute("""
with cadpat as (with locais as (
select
    codigo_do_bem,
    t.codigo_da_localizacao_fisica dest,
    tbo.codigo_da_localizacao_fisica_do_bem orig,
    row_number() over (partition by codigo_do_bem
order by
    tb.codigo_da_transferencia asc) seq
from
    transferencia_bens tb
join tranferencias t on
    tb.codigo_da_transferencia = t.codigo_do_inventario_1
left join transferencia_bens_orig tbo on
    tb.codigo_do_bem_da_transferencia = tbo.codigo_do_bem_da_transferencia
where
    t.codigo_do_tipo_de_transferencia = 4),
grupos as (
select
    dense_rank() over (
    order by identificador) codigo_gru,
    cdtdb.entidade,
    cdtdb.descricao_do_tipo_do_bem nogru_gru,
    identificador codant
from
    cadastro_dos_tipos_de_bem cdtdb),
baixas as (
select
    bb.codigo_do_bem,
    b.data_e_hora_da_baixa::date dtpag_pat,
    b.codigo_do_tipo_de_baixa codigo_bai_pat
from
    baixas_bens bb
join cadastro_de_baixas b on
    bb.codigo_da_baixa = b.identificador_de_registro ),
responsaveis as (
select
    distinct
    dense_rank() over (
    order by identificador_de_registro) codigo_resp,
    p.identificador_de_registro,
    p.nome,
    p.cpf_cnpj cpf
from
    pessoas p
join bens b on
    b.codigo_do_responsavel = p.identificador_de_registro
)
select
    codigo_do_encerramento_do_periodo::int codigo_pat,
    b.codigo_da_entidade::int empresa_pat,
    g.codigo_gru::int,
    to_char(row_number() over (order by codigo_do_encerramento_do_periodo), 'fm000000') chapa_pat,
    null codigo_cpl_pat,
    coalesce(orig, concat(b.codigo_da_entidade, 0)::numeric)::int codigo_set_pat,
    coalesce(b.codigo_da_localizacao_fisica, concat(b.codigo_da_entidade, 0)::numeric)::int codigo_set_atu_pat,
    case
        when classificacao_do_tipo_de_aquisicao = 'M' then 'O'
        when classificacao_do_tipo_de_aquisicao = 'O' then 'I'
        else classificacao_do_tipo_de_aquisicao
    end orig_pat,
    b.codigo_do_grupo_do_bem::int codigo_tip_pat,
    codigo_da_conservacao::int codigo_sit_pat,
    b.descricao_do_bem discr_pat,
    b.data_de_aquisicao_do_bem::date datae_pat,
    coalesce(data_de_inicio_da_depreciacao::date, b.data_de_aquisicao_do_bem::date) dtlan_pat,
    v.valor_convertido_da_aquisicao valaqu_pat,
    v.valor_liquido_contabil valatu_pat,
    b.codigo_do_fornecedor::int codigo_for_pat,
    (v.quantidade_em_anos_de_vida_util * 12)::int percenqtd_pat,
    case
        when quantidade_em_anos_de_vida_util is not null then 'V'
        else 'N'
    end dae_pat,
    v.valor_residual valres_pat,
    case
        when quantidade_em_anos_de_vida_util is not null then 'M'
        else null
    end percentemp_pat,
    nullif(split_part(numero_e_ano_do_empenho, '/', 1), '')::int as nempg_pat,
    substr(case when nullif(split_part(numero_e_ano_do_empenho, '/', 2), '') not like '20%' then to_char(nullif(split_part(numero_e_ano_do_empenho, '/', 2), '')::int, 'fm2000') else nullif(split_part(numero_e_ano_do_empenho, '/', 2), '') end, 1, 4)::int as anoemp_pat,
    b.numero_de_registro_do_bem::bigint codigo_ant_pat,
    numero_da_placa chapa_pat_alt,
    bx.dtpag_pat,
    bx.codigo_bai_pat::int,
    b.numero_do_comprovante nota_pat,
    numero_e_ano_do_processo_administrativo processo_pat,
    r.codigo_resp,
    b.observacao obs_pat
from
    bens b
join valores_dos_bens v on
    v.codigo_do_bem = b.codigo_do_encerramento_do_periodo
left join locais l on
    l.codigo_do_bem = codigo_do_encerramento_do_periodo
    and l.seq = 1
left join cadastro_dos_tipos_de_aquisicao cdt on
    b.codigo_do_tipo_de_aquisicao = cdt.identificador_de_registro
left join grupos g on
    g.codant = codigo_do_tipo_do_bem
left join baixas bx on
    bx.codigo_do_bem = b.codigo_do_encerramento_do_periodo
left join responsaveis r on
    r.identificador_de_registro = b.codigo_do_responsavel)
select
    to_char(codigo_gru, 'fm000') natur,
    chapa_pat,
    a.nome,
    split_part(a.nome, '.', -1) tipo_arq,
    codigo_do_arquivo,
    empresa_pat
from
    cadpat b
join arquivos_do_bem adb on
    b.codigo_pat = adb.codigo_do_bem
join arquivos a on
    a.id = adb.codigo_do_arquivo
--where empresa_pat = 538
order by 1,2
""")

insert = cur_arq.prep("INSERT INTO PATR_ARQUIVOS (CODIGO_ARQ, NATUR, CHAPA, DESCRICAO, TIPO_ARQ, ARQUIVO, EMPRESA) VALUES (?, ?, ?, ?, ?, ?, ?)")
i = 0

for row in cur_orig:
    i += 1
    caminho = os.path.join(pasta_arquivos, row['codigo_do_arquivo'])  # se precisar, + row['tipo_arq']

    with open(caminho, "rb") as f:
        conteudo = f.read()

    cur_arq.execute(insert, (
        i,
        row['natur'],
        row['chapa_pat'],
        to_cp1252_safe(row['nome']),
        row['tipo_arq'],
        conteudo,
        row['empresa_pat']
    ))
    cnx_arq.commit()

NameError: name 'cur_arq' is not defined